# Лабораторная работа № 10 — Теория
## Автоматизация сборки, тестирования и развертывания с помощью CI/CD

### Цель работы
Изучить принципы непрерывной интеграции и непрерывной доставки ПО и научиться:
- объяснять назначение CI/CD и различия между Continuous Integration, Continuous Delivery и Continuous Deployment;
- объяснять назначение Jenkins в процессе автоматизации разработки;
- подключать Git-репозиторий с проектом к Jenkins;
- создавать и настраивать Jenkins Pipeline;
- автоматизировать получение исходного кода из Git;
- выполнять восстановление зависимостей и сборку приложения;
- запускать автоматические тесты в составе Pipeline;
- автоматически создавать Docker-образ приложения;
- использовать Docker-образ для развертывания новой версии приложения в Kubernetes;
- моделировать ошибку сборки/тестирования и анализировать Pipeline.

## 1. Зачем нужен CI/CD

В процессе разработки разработчики регулярно меняют исходный код, добавляют функции, исправляют ошибки и объединяют изменения с общей версией проекта. Если каждое изменение нужно вручную собирать, тестировать, упаковывать и развертывать, процесс становится медленным и подверженным ошибкам. Например, разработчик может:
- изменить исходный код и забыть выполнить сборку;
- не заметить ошибку в одном из модулей;
- создать Docker-образ из неправильной версии приложения;
- развернуть приложение, которое не прошло тестирование.

**CI/CD** (Continuous Integration / Continuous Delivery / Continuous Deployment) — совокупность практик автоматизации разработки, проверки, сборки и доставки программного обеспечения. Они позволяют организовать конвейер, в котором изменения исходного кода автоматически проходят проверки и подготавливаются к развертыванию.

## 2. Continuous Integration (непрерывная интеграция)

**Continuous Integration** — практика регулярного объединения изменений исходного кода с общей версией проекта и автоматической проверки результата.

Последовательность CI:
1. Разработчик изменяет исходный код.
2. Создает commit.
3. Отправляет изменения в Git-репозиторий.
4. Система автоматизации получает новую версию кода.
5. Выполняется восстановление зависимостей (`restore`).
6. Выполняется сборка проекта (`build`).
7. Запускаются автоматические тесты (`test`).
8. Формируется результат проверки.

Если сборка или тестирование завершились ошибкой, разработчик получает информацию о проблеме и исправляет ее до дальнейшей доставки приложения.

## 3. Continuous Delivery и Continuous Deployment

**Continuous Delivery (непрерывная доставка)** — практика автоматической подготовки проверенной версии приложения к развертыванию. После успешной сборки и тестирования приложение упаковывается в готовый артефакт, например Docker-образ. Сам запуск развертывания может требовать дополнительного подтверждения.

**Continuous Deployment (непрерывное развертывание)** — практика, при которой изменения, успешно прошедшие все проверки, автоматически развертываются в целевой среде без ручного подтверждения.

| Практика | Что автоматизировано | Ручное подтверждение перед выпуском |
|---|---|---|
| Continuous Integration | сборка и тесты при каждом изменении | не относится к выпуску |
| Continuous Delivery | + подготовка проверенного артефакта | может требоваться |
| Continuous Deployment | + автоматическое развертывание | не требуется |

В лабораторной работе настраивается цепочка, в которой после успешной сборки и тестирования создается Docker-образ и выполняется развертывание в Kubernetes. Поэтому практическая реализация демонстрирует CI и автоматизированное развертывание, соответствующее подходу **Continuous Deployment**.

## 4. Pipeline (конвейер)

**Pipeline** — последовательность автоматизированных этапов, которые выполняются для обработки изменений исходного кода. Каждый этап отвечает за определенную операцию.

В ЛР №10 используется следующая последовательность:

| Этап | Назначение |
|---|---|
| Checkout | Получение исходного кода из Git |
| Restore | Восстановление зависимостей .NET |
| Build | Сборка проекта |
| Test | Запуск автоматических тестов |
| Package | Создание Docker-образа |
| Deploy | Развертывание новой версии в Kubernetes |

**Правило перехода между этапами.** Переход к следующему этапу выполняется только после успешного завершения предыдущего обязательного этапа. Если Unit-тесты завершились ошибкой, Pipeline не переходит к созданию Docker-образа и развертыванию. Так в целевую среду не попадает версия, которая не прошла проверки.

## 5. Jenkins

**Jenkins** — сервер автоматизации для задач сборки, тестирования и доставки ПО. Он позволяет создавать задания, которые запускаются вручную или автоматически при наступлении событий.

С помощью Jenkins можно:
- получать исходный код из Git;
- запускать команды сборки;
- выполнять автоматические тесты;
- создавать Docker-образы;
- взаимодействовать с Kubernetes;
- сохранять результаты выполнения заданий;
- отображать ошибки и журналы;
- запускать последовательности действий в виде Pipeline.

В ЛР Jenkins — центральный компонент автоматизации. Разработчик не выполняет команды после каждого изменения вручную: он отправляет изменения в Git, а Jenkins выполняет настроенную последовательность действий.

**Jenkins Job** — задание, которое Jenkins выполняет по заданным правилам.
**Jenkins Pipeline** — описание последовательности этапов выполнения задания.

## 6. Способы создания Pipeline и Jenkinsfile

Pipeline можно определить двумя способами:
- создать Pipeline непосредственно в интерфейсе Jenkins;
- хранить описание Pipeline в файле **Jenkinsfile** в Git-репозитории.

В ЛР №10 используется второй способ. **Jenkinsfile** — текстовый файл с описанием Pipeline. Он хранится вместе с исходным кодом и позволяет версионировать сам процесс автоматизации.

Структура одного этапа:

In [ ]:
# Структура Jenkinsfile (Groovy, декларативный синтаксис) — приведена как текст, не исполняется
jenkinsfile_skeleton = """
pipeline {
    agent any                  // Jenkins может использовать любой доступный исполнитель
    stages {                   // набор этапов
        stage('Test') {        // отдельный этап
            steps {            // команды, выполняемые на этапе
                // запуск тестов
            }
        }
    }
}
"""
print(jenkinsfile_skeleton)

Элементы Jenkinsfile:
- `pipeline` — начало описания Pipeline;
- `agent any` — Jenkins может использовать любой доступный исполнитель;
- `stages` — набор этапов;
- `stage` — отдельный этап;
- `steps` — команды, которые выполняются на этапе, в заданном порядке.

## 7. Git в процессе CI/CD

Git хранит исходный код приложения и управляет изменениями. Jenkins получает код непосредственно из Git-репозитория.

Сценарий работы:
1. Изменяется C#-проект и сохраняются изменения.
2. Создается commit.
3. Изменения отправляются в Git-репозиторий.
4. Jenkins получает актуальную версию проекта.
5. Запускается Pipeline.

Способы запуска задания:
- **ручной запуск**;
- **периодическая проверка репозитория** (Poll SCM) — Jenkins по расписанию проверяет, появились ли новые изменения, и если да, запускает задание;
- **Webhook** — Git-сервис (например, GitHub) отправляет Jenkins уведомление сразу после нового commit.

В рамках ЛР допускаются ручной запуск и Poll SCM. Пример расписания Poll SCM: `H/2 * * * *` — проверка примерно раз в две минуты; точное время определяет планировщик Jenkins.

## 8. .NET SDK: restore, build, test, publish

| Команда | Назначение |
|---|---|
| `dotnet restore` | Восстановление зависимостей проекта |
| `dotnet build` | Сборка проекта |
| `dotnet test` | Запуск автоматических тестов |
| `dotnet publish` | Подготовка приложения к публикации |

Типичные команды в Pipeline:

```bash
dotnet restore
dotnet build --configuration Release
dotnet test --configuration Release
```

Если проект состоит из нескольких частей, команды нужно выполнять для соответствующих проектов или для файла решения `.sln`.

## 9. Тестирование в CI/CD

**Unit Testing (модульное тестирование)** — проверка отдельных элементов программы: методов и классов. В CI/CD unit-тесты автоматически проверяют корректность логики после получения новых изменений. К автоматическим тестам также относятся интеграционные и нагрузочные тесты.

Jenkins анализирует код возврата команды. Если команда завершилась ошибкой, Pipeline получает статус **Failed** и прекращает выполнение последующих этапов. Поэтому ошибка тестирования препятствует созданию и развертыванию новой версии.

Результаты тестов можно сохранить в формате TRX:

```bash
dotnet test Tests/Tests.csproj --configuration Release --logger "trx"
```

## 10. Docker-образ

После успешной сборки и тестирования приложение упаковывается в Docker-образ:

```bash
docker build -t riot-server:1.0 .
```

В ЛР допускается использовать номер сборки Jenkins (`BUILD_NUMBER`) как тег образа. Уникальный тег позволяет однозначно связать образ с конкретным запуском Pipeline.

## 11. Развертывание в Kubernetes

В ЛР №9 были созданы Deployment и Service. В ЛР №10 развертывание обновляется автоматически: после успешного создания образа Jenkins выполняет команду обновления образа в Deployment:

```bash
kubectl set image deployment/riot-server server=riot-server:15
```

Здесь:
- `deployment/riot-server` — имя Deployment;
- `server` — имя контейнера внутри Deployment;
- `riot-server:15` — новый Docker-образ.

Проверка результата:

```bash
kubectl rollout status deployment/riot-server
kubectl get deployments
kubectl get pods
```

`kubectl rollout status` ожидает завершения обновления и возвращает результат. Это позволяет Jenkins понять, успешно ли прошло развертывание.

**Важно:** образ, собранный на машине разработчика, не обязательно доступен Kubernetes. Если кластер работает в Docker Desktop, он использует тот же Docker Engine и видит образ. Если используется Minikube, образ нужно загрузить в его среду: `minikube image load riot-server:15`.

## 12. Проверка поведения Pipeline при ошибках

| Ситуация | Ожидаемое поведение |
|---|---|
| Проект успешно собирается | Pipeline переходит к тестированию |
| Сборка завершилась ошибкой | Pipeline останавливается |
| Unit-тесты завершились ошибкой | Pipeline не переходит к созданию Docker-образа |
| Docker-образ успешно создан | Pipeline может перейти к развертыванию |
| Развертывание завершилось ошибкой | Pipeline получает статус Failed |
| Все этапы завершились успешно | Pipeline получает статус Success |

Для моделирования ошибки временно меняют тест так, чтобы он падал (например, изменяют ожидаемое значение в `Assert`). После эксперимента тест восстанавливают, отправляют исправление в Git и повторно запускают Pipeline.

## 13. Контрольные вопросы

1. Что такое Continuous Integration?
2. Что такое Continuous Delivery?
3. Чем Continuous Delivery отличается от Continuous Deployment?
4. Что такое Pipeline?
5. Для чего используется Jenkins?
6. Что такое Jenkins Job?
7. Чем Jenkins Pipeline отличается от обычного Jenkins Job?
8. Что такое Jenkinsfile?
9. Для чего используется этап Restore?
10. Почему тесты необходимо запускать до создания Docker-образа?
11. Что происходит, если тесты завершаются ошибкой?
12. Что выполняется на этапе Package?
13. Для чего используется этап Deploy?
14. Как Jenkins может обновить Docker-образ в Kubernetes Deployment?
15. Для чего используется команда `kubectl rollout status`?
16. Почему Jenkins должен иметь доступ к Kubernetes?
17. Что такое Poll SCM?
18. Что может привести к неуспешному выполнению Jenkins Pipeline?